# 🤖 Coding Agent MVP1 on Google Colab (free, no API key)

A small **Claude Code–style coding agent** that runs entirely in Google Colab. It can **create, read, edit and list files**, **run shell commands**, and manage **background jobs** (e.g. a Flask server) inside a workspace folder, and it asks for your approval (`y/N`) before anything is written or run.

✅ **Tested and working on Colab** with a free **Tesla T4 GPU** and the open model **`qwen2.5:7b`** served by **Ollama**. No paid API, no Hugging Face credits, no API key needed.

**Stack:** LangChain + LangGraph agent · Ollama (local LLM on the Colab GPU) · human-in-the-loop approval · audit log · command safety checks.

> Based on `5.CodingAgentMVP1` from [singhsanket143/AiEngineeringConcepts](https://github.com/singhsanket143/AiEngineeringConcepts), adapted to run on free models in Colab.

### How to use
1. **Runtime → Change runtime type → T4 GPU**
2. Run the cells below **in order** (first run takes a few minutes to download the model)
3. Chat with the agent in step 3, approve actions with `y`, type `exit` to stop

Project files are kept in your Google Drive at `MyDrive/CodingAgentMVP1_HF` (upload `CodingAgentMVP1_HF.zip` if prompted).

## 1️⃣ Setup: mount Drive, find the project, install requirements
Also patches `src/models.py` so the agent can talk to any OpenAI-compatible endpoint (local Ollama, Groq, OpenRouter, ...).

In [ ]:
# 1) Find project, install requirements, and patch models.py to support any OpenAI-compatible endpoint
import os, zipfile
from google.colab import drive, files

drive.mount('/content/drive')
CANDIDATES = ['/content/drive/MyDrive/CodingAgentMVP1_HF', '/content/CodingAgentMVP1_HF']
PROJECT = next((p for p in CANDIDATES if os.path.isdir(p + '/src')), None)
if PROJECT is None:
    print("Project not found - upload CodingAgentMVP1_HF.zip")
    up = files.upload()
    zipfile.ZipFile(list(up)[0]).extractall('/content')
    PROJECT = '/content/CodingAgentMVP1_HF'
print("Project:", PROJECT)
%cd {PROJECT}
!pip install -q -r requirements.txt

MODELS_PY = r'''import os
from dotenv import load_dotenv
from dataclasses import dataclass
from langchain_openai import ChatOpenAI


load_dotenv()


@dataclass(frozen=True)
class Provider:
    name: str
    env_var: str
    is_free: bool
    base_url: str | None
    model: str


def _providers() -> list[Provider]:
    providers: list[Provider] = []

    # Any OpenAI-compatible endpoint: local Ollama, Groq, OpenRouter, vLLM, ...
    if os.getenv("CUSTOM_BASE_URL"):
        providers.append(Provider(
            os.getenv("CUSTOM_NAME", "Custom"),
            "CUSTOM_API_KEY",
            True,
            os.getenv("CUSTOM_BASE_URL"),
            os.getenv("CUSTOM_MODEL", ""),
        ))

    providers.append(Provider(
        "HuggingFace", "HF_TOKEN", True,
        "https://router.huggingface.co/v1",
        os.getenv("HF_MODEL", "meta-llama/Llama-3.1-8B-Instruct"),
    ))
    providers.append(Provider("OpenAI", "OPENAI_API_KEY", False, None, "gpt-4o-mini"))
    return providers


def select_provider() -> Provider:
    for provider in _providers():
        if os.getenv(provider.env_var):
            return provider
    raise RuntimeError("No provider found. Set CUSTOM_BASE_URL+CUSTOM_API_KEY, HF_TOKEN or OPENAI_API_KEY.")


def build_chat_model() -> tuple[ChatOpenAI, Provider]:
    provider = select_provider()
    kwargs: dict = {
        "model": provider.model,
        "api_key": os.getenv(provider.env_var),
        "temperature": 0.1,
    }
    if provider.base_url is not None:
        kwargs["base_url"] = provider.base_url
    return ChatOpenAI(**kwargs), provider
'''
open(PROJECT + '/src/models.py', 'w').write(MODELS_PY)
print("models.py patched")

## 2️⃣ Choose the backend (default: free model on the Colab GPU)
- `MODE = "ollama"` → installs Ollama and pulls `qwen2.5:7b` onto the Colab T4. **Free, no key.** *(default, tested)*
- `MODE = "groq"` / `"openrouter"` → free hosted models; needs a key saved in 🔑 Secrets (`GROQ_API_KEY` / `OPENROUTER_API_KEY`).
- Change `OLLAMA_MODEL` to `qwen2.5:3b` if you run out of GPU memory or want it faster (weaker tool calling).

Ollama and the model are re-downloaded each time the Colab session restarts. Your project files stay on Drive.

In [ ]:
# 2) Pick the backend
import os, subprocess, time, requests
MODE = "ollama"     # "ollama" (free, on Colab GPU) | "groq" | "openrouter"
OLLAMA_MODEL = "qwen2.5:7b"   # use "qwen2.5:3b" if memory/speed is a problem

os.environ['WORK_DIR'] = PROJECT + '/workspace'
os.environ['HITL_ENABLED'] = 'true'          # asks y/N before write / edit / run
os.environ['MAX_MODEL_CALLS_PER_RUN'] = '15'
for k in ['HF_TOKEN', 'OPENAI_API_KEY']:
    os.environ.pop(k, None)

if MODE == "ollama":
    !nvidia-smi -L || echo "No GPU! Runtime -> Change runtime type -> T4 GPU"
    !apt-get -qq install -y zstd > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
    subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(30):
        try:
            requests.get("http://localhost:11434"); break
        except Exception:
            time.sleep(1)
    !ollama pull {OLLAMA_MODEL}
    os.environ.update(CUSTOM_NAME="Ollama (Colab GPU)", CUSTOM_BASE_URL="http://localhost:11434/v1",
                      CUSTOM_API_KEY="ollama", CUSTOM_MODEL=OLLAMA_MODEL)
elif MODE == "groq":
    from google.colab import userdata
    os.environ.update(CUSTOM_NAME="Groq", CUSTOM_BASE_URL="https://api.groq.com/openai/v1",
                      CUSTOM_API_KEY=userdata.get('GROQ_API_KEY').strip(), CUSTOM_MODEL="llama-3.3-70b-versatile")
elif MODE == "openrouter":
    from google.colab import userdata
    os.environ.update(CUSTOM_NAME="OpenRouter", CUSTOM_BASE_URL="https://openrouter.ai/api/v1",
                      CUSTOM_API_KEY=userdata.get('OPENROUTER_API_KEY').strip(),
                      CUSTOM_MODEL="meta-llama/llama-3.3-70b-instruct:free")
print("Backend ready:", MODE, os.environ['CUSTOM_MODEL'])

## 3️⃣ Chat with the agent
Try:
- `List the files in the workspace`
- `Create binary_search.py with a binary search function and run it`
- `Create greet.js that exports a function greet(name)`
- `Run the flask app in app.py in the background`

The agent shows exactly what it will write or run and waits for **`y`** before doing it. Type **`exit`** to stop.

*Tip: small models sometimes ask "shall I proceed?" before acting; just answer `yes`.*

In [ ]:
# 3) Start the agent (type 'exit' to stop)
import sys
sys.path.insert(0, PROJECT + '/src')
for mod in ['models', 'agent', 'main']:
    sys.modules.pop(mod, None)
import main
main.chat()

## 4️⃣ Look at the files the agent created
Files appear in `workspace/` (also visible in the Colab file browser and your Google Drive).

In [ ]:
# 4) See what the agent created
!ls -R "$WORK_DIR"

## ✅ Verified run
On a Colab T4 with `qwen2.5:7b` the agent answered chat messages, proposed a binary search implementation, showed the human-in-the-loop approval prompt for `write_file`, and wrote `binary_search.txt` into the workspace on Google Drive.

## 🛠 Troubleshooting
| Problem | Fix |
|---|---|
| Very slow / no GPU message | Runtime → Change runtime type → **T4 GPU** |
| Out of memory | set `OLLAMA_MODEL = "qwen2.5:3b"` in step 2 |
| Agent replies with text but doesn't use tools | Try a larger model, or `MODE = "groq"` |
| `402 no remaining credits` (Hugging Face router) | HF free credits are used up; use the default `ollama` mode |
| Changed model but nothing changes | Re-run step 2, then step 3 (it reloads the modules) |